# PRISM Theme 5 — FDB-v3 100-sample evaluation (Colab GPU)

Upload `PRISM_Z_Colab.zip` next to this notebook, then Run all.

**What it does:** installs the LiveKit+NeMo stack, unpacks the patched benchmark + PRISM agent, downloads the FDB-v3 data, starts the PRISM agent worker, runs all 100 examples (GPU Parakeet ASR), evaluates, and zips results for download.

**You need:** free LiveKit Cloud credentials + a Gemini API key (paste when asked — never stored in the notebook).

**Runtime:** use a GPU runtime (T4 is fine). Total ≈ 1.5–2.5 h depending on Gemini rate limits.

In [ ]:
# @title 1. Check GPU + unzip project bundle
!nvidia-smi -L || echo "WARNING: no GPU attached — switch runtime to GPU"
!unzip -q -o PRISM_Z_Colab.zip -d /content/
!ls /content/prism_colab/

In [ ]:
# @title 2. Install stack (5–10 min)
!apt-get -qq install -y ffmpeg > /dev/null
%cd /content
!pip install -q "livekit-agents[google]" "livekit-plugins-silero" "livekit[crypto]" numpy python-dotenv pydub ffmpeg-python gdown
!pip install -q "nemo_toolkit[asr]"
print("stack installed")

In [ ]:
# @title 3. Unpack benchmark code, apply patches, download data (~750 MB, one-time)
import os, subprocess
os.chdir("/content")
if not os.path.exists("/content/Full-Duplex-Bench"):
    subprocess.run(["git","clone","--depth","1","https://github.com/DanielLin94144/Full-Duplex-Bench.git"], check=True)
# copy patched files from the bundle over the fresh clone
src = "/content/prism_colab/v3"
dst = "/content/Full-Duplex-Bench/v3"
subprocess.run(["cp","-r",src+"/.",dst+"/"], check=True)
if not os.path.exists(dst+"/fdb_v3_data_released"):
    subprocess.run(["gdown","1SO_4MTazWQ_jvCx0dtmpQ-t40bdd07yz","-O","/content/fdb_v3_data.zip"], check=True)
    import zipfile
    with zipfile.ZipFile("/content/fdb_v3_data.zip") as z:
        z.extractall("/content/Full-Duplex-Bench/v3")
print("examples:", len(os.listdir(dst+"/fdb_v3_data_released")))

In [ ]:
# @title 4. Credentials (entered interactively, never printed or saved to the notebook)
import getpass, os
os.makedirs("/content/Full-Duplex-Bench/v3/logs", exist_ok=True)
env = f"""LIVEKIT_URL={getpass.getpass('LIVEKIT_URL (wss://...): ')}
LIVEKIT_API_KEY={getpass.getpass('LIVEKIT_API_KEY: ')}
LIVEKIT_API_SECRET={getpass.getpass('LIVEKIT_API_SECRET: ')}
GOOGLE_API_KEY={getpass.getpass('GOOGLE_API_KEY (Gemini): ')}
"""
open("/content/Full-Duplex-Bench/v3/.env.local","w").write(env)
print(".env.local written (gitignored upstream)")

In [ ]:
# @title 5. Start PRISM agent worker (supervised background loop)
import subprocess, time, os
os.chdir("/content/Full-Duplex-Bench/v3")
env = dict(os.environ, LK_PROVIDER="gemini2_5", HF_HOME="/content/hf")
# Supervised: upstream livekit_ffi has a soxr FFT-cache race that can
# abort the worker after its first job; this loop restarts it
# automatically and the batch runner resumes (skips completed results).
sup = subprocess.Popen(
    "while [ ! -f STOP_WORKER ]; do python -u prism_agent.py start; echo '[sup] worker exited '$?'; sleep 3; done",
    shell=True, env=env, stdout=open("/content/worker.log","w"), stderr=subprocess.STDOUT)
time.sleep(30)
!tail -3 /content/worker.log
print("supervisor pid:", sup.pid)

In [ ]:
# @title 6. Run the 100-example benchmark (1.5–2.5 h; resumable — just re-run this cell if interrupted)
import subprocess, os
os.chdir("/content/Full-Duplex-Bench/v3")
env = dict(os.environ, HF_HOME="/content/hf")
!echo "--- progress: $(ls fdb_v3_data_released/*/result_prism.json 2>/dev/null | wc -l)/100 done ---"
subprocess.run(["python","run_tool_benchmark_all_released.py","--provider","prism","--root_dir","fdb_v3_data_released"], env=env)

In [ ]:
# @title 7. Evaluate (tool F1, pass rate, latency)
import subprocess, os
os.chdir("/content/Full-Duplex-Bench/v3")
B="--benchmark benchmark_data_v2.json --results-dir fdb_v3_data_released --provider prism"
subprocess.run(f"python evaluate_tool_calls.py {B} --output prism_evaluation_report.json", shell=True)
subprocess.run(f"python evaluate_pass_rate.py {B} --output prism_pass_rate_report.json", shell=True)
subprocess.run(f"python analyze_tool_latency.py --results-dir fdb_v3_data_released --provider prism", shell=True)
import json
r = json.load(open("prism_pass_rate_report.json"))
print("\n=== STRICT PASS RATE:", r["overall_pass_rate"], f"({r['passed']}/{r['total_scenarios']}) ===")

In [ ]:
# @title 8. Package results (JSONs + reports only, no WAVs) and download
import zipfile, os, glob
os.chdir("/content/Full-Duplex-Bench/v3")
with zipfile.ZipFile("/content/prism_results.zip","w",zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob("fdb_v3_data_released/*/result_prism.json"):
        z.write(f)
    for f in glob.glob("prism_*.json") + glob.glob("logs/*.log"):
        z.write(f)
print("size MB:", round(os.path.getsize("/content/prism_results.zip")/1e6,1))
from google.colab import files
files.download("/content/prism_results.zip")